# UIT DSC 2026 - LegalQA RAG (Standalone Colab)

Notebook này chứa toàn bộ code RAG trực tiếp. GitHub chỉ được dùng để tải dữ liệu
văn bản đã xử lý; không chạy bất kỳ file Python nào từ repository.

Chọn **Runtime > Change runtime type > GPU** rồi chạy lần lượt từ trên xuống.



In [ ]:
import gc
import hashlib
import json
import math
import os
import re
import shutil
import unicodedata
import zipfile
from pathlib import Path

import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Hãy bật GPU trong Runtime > Change runtime type"
print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi



## 1. Cài dependencies



In [ ]:
!pip install -q "numpy>=1.26" "transformers>=4.46" "sentence-transformers>=3.0" "accelerate>=1.0" "faiss-cpu>=1.8" "bitsandbytes>=0.43"



## 2. Cấu hình

Đồng bộ với pipeline đã chạy: build batch 32, query batch 16, rerank batch 1 và generation 4-bit.



In [ ]:
WORK_DIR = Path("/content/legal-rag")
DATA_DIR = WORK_DIR / "data" / "documents"
ARTIFACT_DIR = WORK_DIR / "artifacts"
OUTPUT_DIR = WORK_DIR / "outputs"
NODES_PATH = ARTIFACT_DIR / "nodes.jsonl"
INDEX_PATH = ARTIFACT_DIR / "index.faiss"
MANIFEST_PATH = ARTIFACT_DIR / "index.manifest.json"

EMBED_MODEL_ID = "AITeamVN/Vietnamese_Embedding"
RERANK_MODEL_ID = "AITeamVN/Vietnamese_Reranker"
GENERATOR_MODEL_ID = "AITeamVN/Vi-Qwen2-3B-RAG"

EMBED_BATCH = 32
QUERY_BATCH = 16
RERANK_BATCH = 1
DENSE_TOP_K = 50
RERANK_TOP_K = 30
CONTEXT_TOP_K = 6
MAX_CHUNK_WORDS = 900
MAX_CONTEXT_WORDS = 5000
MAX_NEW_TOKENS = 1024
USE_4BIT = True

for directory in (DATA_DIR, ARTIFACT_DIR, OUTPUT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Workspace:", WORK_DIR)



## 3. Lấy dữ liệu từ GitHub

Cell này chỉ tải archive và giải nén `outputs/cleaned/documents`. Không import hoặc
chạy code từ GitHub. Nếu dữ liệu đã có trong `DATA_DIR`, cell sẽ bỏ qua.



In [ ]:
import requests

ARCHIVE_URL = "https://github.com/lengoc-tuyen/DSC-task2/archive/refs/heads/main.zip"
ARCHIVE_PATH = WORK_DIR / "github-data.zip"

if not any(DATA_DIR.glob("context_*.json")):
    if not ARCHIVE_PATH.exists():
        print("Đang tải archive dữ liệu từ GitHub...")
        with requests.get(ARCHIVE_URL, stream=True, timeout=120) as response:
            response.raise_for_status()
            with ARCHIVE_PATH.open("wb") as stream:
                for block in response.iter_content(8 * 1024 * 1024):
                    if block:
                        stream.write(block)
    print("Đang giải nén documents...")
    with zipfile.ZipFile(ARCHIVE_PATH) as archive:
        members = [
            name for name in archive.namelist()
            if "/outputs/cleaned/documents/context_" in name and name.endswith(".json")
        ]
        assert members, "Archive không chứa outputs/cleaned/documents"
        for index, member in enumerate(members, 1):
            target = DATA_DIR / Path(member).name
            if not target.exists():
                with archive.open(member) as source, target.open("wb") as destination:
                    shutil.copyfileobj(source, destination)
            if index % 1000 == 0:
                print(f"Đã giải nén {index:,}/{len(members):,} files")

documents = sorted(DATA_DIR.glob("context_*.json"))
print("Số văn bản:", len(documents))
assert len(documents) >= 8500, "Thiếu dữ liệu văn bản"



## 4. Code tạo article-family chunks



In [ ]:
def document_label(metadata):
    return " ".join(
        value for value in (metadata.get("document_type"), metadata.get("document_number")) if value
    ) or metadata.get("name")


def legal_label(kind, number, title):
    return f"{kind} {number or ''}{': ' + title if title else ''}".strip()


def split_oversize_units(units, max_words):
    expanded = []
    for content, clause, points in units:
        words = (content or "").split()
        if len(words) <= max_words:
            expanded.append((content, clause, points))
        else:
            for start in range(0, len(words), max_words):
                expanded.append((" ".join(words[start:start + max_words]), clause, points))
    return expanded


def group_units(units, max_words, overlap=1):
    chunks, current, word_count = [], [], 0
    for unit in units:
        size = len((unit[0] or "").split())
        if current and word_count + size > max_words:
            chunks.append(current)
            current = current[-min(overlap, len(current)):] if overlap else []
            word_count = sum(len((item[0] or "").split()) for item in current)
            if current and word_count + size > max_words:
                current, word_count = [], 0
        current.append(unit)
        word_count += size
    if current:
        chunks.append(current)
    return chunks


def make_node(node_id, document, part, node_type, content, path, **extra):
    return {
        "node_id": str(node_id),
        "document_id": int(document["document_id"]),
        "part_id": str(part.get("part_id") or ""),
        "node_type": node_type,
        "content": content.strip(),
        "path": list(path),
        "metadata": document.get("metadata") or {},
        **extra,
    }


def article_nodes(document, part, article, path, max_words=900, overlap=1):
    article_path = (*path, legal_label("Điều", article.get("number"), article.get("title")))
    units = []
    if article.get("content"):
        units.append((article["content"], None, ()))
    for clause in article.get("clauses") or []:
        content = "\n".join(
            value for value in [
                clause.get("content"),
                *[point.get("content") for point in clause.get("points") or []],
            ] if value
        )
        points = tuple(point.get("label") for point in clause.get("points") or [] if point.get("label"))
        units.append((content, clause.get("number"), points))
    for point in article.get("points") or []:
        units.append((point.get("content"), None, (point.get("label"),)))

    chunks = group_units(split_oversize_units(units, max_words), max_words, overlap)
    for ordinal, chunk in enumerate(chunks, 1):
        content = "\n".join(unit[0] for unit in chunk if unit[0])
        clauses = [unit[1] for unit in chunk if unit[1]]
        points = [label for unit in chunk for label in unit[2] if label]
        chunk_path = (*article_path, *(f"Khoản {number}" for number in clauses))
        yield make_node(
            f"{article.get('article_id')}:retrieval:{ordinal}", document, part,
            "article_chunk", content, chunk_path,
            article_id=article.get("article_id"),
            article_number=article.get("number"),
            clause_numbers=clauses,
            point_labels=points,
        )


def block_nodes(document, part, block, path, max_words=900):
    words = (block.get("content") or "").split()
    for ordinal, start in enumerate(range(0, len(words), max_words), 1):
        content = " ".join(words[start:start + max_words])
        yield make_node(
            f"{block.get('block_id')}:retrieval:{ordinal}", document, part,
            "unstructured", content, path,
            article_id=None, article_number=None, clause_numbers=[], point_labels=[],
        )


def section_nodes(document, part, section, path):
    section_path = (*path, legal_label("Mục", section.get("number"), section.get("title")))
    for article in section.get("articles") or []:
        yield from article_nodes(document, part, article, section_path, MAX_CHUNK_WORDS)


def document_nodes(document):
    if document.get("status") == "failed":
        return
    metadata = document.get("metadata") or {}
    root_path = tuple(value for value in (document_label(metadata), metadata.get("title")) if value)
    for part in (document.get("tree") or {}).get("parts") or []:
        part_path = (*root_path, *((part.get("part_title"),) if part.get("part_title") else ()))
        for block in part.get("unstructured_blocks") or []:
            yield from block_nodes(document, part, block, part_path, MAX_CHUNK_WORDS)
        for article in part.get("articles") or []:
            yield from article_nodes(document, part, article, part_path, MAX_CHUNK_WORDS)
        for section in part.get("sections") or []:
            yield from section_nodes(document, part, section, part_path)
        for chapter in part.get("chapters") or []:
            chapter_path = (*part_path, legal_label("Chương", chapter.get("number"), chapter.get("title")))
            for article in chapter.get("articles") or []:
                yield from article_nodes(document, part, article, chapter_path, MAX_CHUNK_WORDS)
            for section in chapter.get("sections") or []:
                yield from section_nodes(document, part, section, chapter_path)


def embedding_text(node):
    return "\n".join([*node.get("path", []), node.get("content", "")])


def iter_nodes(path=NODES_PATH):
    with path.open(encoding="utf-8") as stream:
        for line in stream:
            if line.strip():
                yield json.loads(line)



## 5. Xuất nodes.jsonl



In [ ]:
if NODES_PATH.exists():
    print("Đã có nodes:", NODES_PATH)
else:
    temporary = NODES_PATH.with_suffix(".jsonl.tmp")
    count = 0
    with temporary.open("w", encoding="utf-8") as output:
        for doc_index, path in enumerate(documents, 1):
            document = json.loads(path.read_text(encoding="utf-8"))
            for node in document_nodes(document):
                output.write(json.dumps(node, ensure_ascii=False, separators=(",", ":")) + "\n")
                count += 1
            if doc_index % 500 == 0:
                print(f"Documents {doc_index:,}/{len(documents):,}; nodes {count:,}")
    temporary.replace(NODES_PATH)
    print("Đã tạo nodes:", count)

print(f"Nodes size: {NODES_PATH.stat().st_size / 1024**2:.1f} MB")



## 6. Build FAISS IndexFlatIP trực tiếp



In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


def encode_normalized(model, texts, batch_size):
    vectors = np.asarray(model.encode(
        list(texts), batch_size=batch_size, convert_to_numpy=True,
        show_progress_bar=False,
    ), dtype=np.float32)
    if vectors.ndim != 2 or not np.isfinite(vectors).all():
        raise ValueError("Embedding model trả vector không hợp lệ")
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError("Embedding model trả zero vector")
    return np.ascontiguousarray(vectors / norms, dtype=np.float32)


if INDEX_PATH.exists() and MANIFEST_PATH.exists():
    print("FAISS index đã tồn tại, bỏ qua build.")
else:
    embedder = SentenceTransformer(EMBED_MODEL_ID, device="cuda")
    embedder.max_seq_length = 2048
    index = None
    texts, encoded = [], 0
    for node in iter_nodes():
        texts.append(embedding_text(node))
        if len(texts) >= EMBED_BATCH:
            vectors = encode_normalized(embedder, texts, EMBED_BATCH)
            if index is None:
                index = faiss.IndexFlatIP(vectors.shape[1])
            index.add(vectors)
            encoded += len(texts)
            texts = []
            if encoded % (EMBED_BATCH * 100) == 0:
                print(f"Embedded {encoded:,} nodes", flush=True)
    if texts:
        vectors = encode_normalized(embedder, texts, EMBED_BATCH)
        if index is None:
            index = faiss.IndexFlatIP(vectors.shape[1])
        index.add(vectors)
        encoded += len(texts)
    assert index is not None and index.ntotal == encoded
    temporary_index = INDEX_PATH.with_suffix(".faiss.tmp")
    faiss.write_index(index, str(temporary_index))
    temporary_index.replace(INDEX_PATH)
    digest = hashlib.sha256()
    with NODES_PATH.open("rb") as stream:
        while block := stream.read(1024 * 1024):
            digest.update(block)
    manifest = {
        "index_type": "IndexFlatIP", "model": EMBED_MODEL_ID,
        "dimension": index.d, "node_count": index.ntotal,
        "nodes_sha256": digest.hexdigest(),
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("Build hoàn tất:", manifest)



## 7. Upload file đề

Chọn file JSON có 1.000 mẫu dạng `{"id": {"question": "...", "answer": null}}`.



In [ ]:
from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, "Chỉ upload một file đề JSON"
uploaded_name = next(iter(uploaded))
QUESTIONS_PATH = WORK_DIR / "public-official.json"
shutil.move(uploaded_name, QUESTIONS_PATH)
questions_data = json.loads(QUESTIONS_PATH.read_text(encoding="utf-8"))
assert isinstance(questions_data, dict), "File đề phải là JSON object"
assert len(questions_data) == 1000, f"Đề phải có 1.000 câu, hiện có {len(questions_data)}"
assert all(
    isinstance(record, dict) and isinstance(record.get("question"), str) and record["question"].strip()
    for record in questions_data.values()
), "Mỗi ID phải có question không rỗng"
questions = {str(qid): record["question"].strip() for qid, record in questions_data.items()}
print("Đã đọc", len(questions), "câu")



## 8. Phân tích câu hỏi và dense retrieval



In [ ]:
ARTICLE_RE = re.compile(r"\bĐiều\s+(\d+[a-zđ]?)\b", re.IGNORECASE)
CLAUSE_RE = re.compile(r"\bkhoản\s+(\d+)\b", re.IGNORECASE)
DOCUMENT_RE = re.compile(r"\b\d{1,4}/\d{4}/[A-ZĐ0-9-]+\b", re.IGNORECASE)


def analyze_question(question):
    normalized = re.sub(r"\s+", " ", unicodedata.normalize("NFC", question)).strip()
    folded = normalized.casefold()
    if any(term in folded for term in ("xử phạt", "mức phạt", "phạt thế nào", "bị phạt")):
        hint = "mức phạt hình thức xử phạt biện pháp khắc phục"
    elif any(term in folded for term in ("thủ tục", "hồ sơ", "hướng dẫn")):
        hint = "trình tự thủ tục hồ sơ"
    elif any(term in folded for term in ("thời hạn", "bao lâu", "thời gian")):
        hint = "thời hạn thời gian"
    elif any(term in folded for term in ("điều kiện", "đối tượng nào")):
        hint = "điều kiện đối tượng"
    else:
        hint = ""
    return normalized, f"{normalized} {hint}".strip()


nodes = list(iter_nodes())
index = faiss.read_index(str(INDEX_PATH))
assert index.ntotal == len(nodes), "Số dòng FAISS không khớp nodes"

# Model embedding có thể vẫn còn từ bước build; nếu runtime mới thì tải lại.
if "embedder" not in globals():
    embedder = SentenceTransformer(EMBED_MODEL_ID, device="cuda")
    embedder.max_seq_length = 2048

analyses = [(qid, question, *analyze_question(question)) for qid, question in questions.items()]
query_vectors = encode_normalized(embedder, [item[3] for item in analyses], QUERY_BATCH)
scores, rows = index.search(query_vectors, min(DENSE_TOP_K, len(nodes)))

candidates_path = OUTPUT_DIR / "candidates.jsonl"
candidates_tmp = candidates_path.with_suffix(".jsonl.tmp")
with candidates_tmp.open("w", encoding="utf-8") as output:
    for (qid, original, _, _), row_scores, row_ids in zip(analyses, scores, rows):
        hits = [
            {"node_id": nodes[int(row)]["node_id"], "dense_score": float(score)}
            for row, score in zip(row_ids, row_scores) if row >= 0
        ]
        output.write(json.dumps({"id": qid, "question": original, "hits": hits}, ensure_ascii=False) + "\n")
candidates_tmp.replace(candidates_path)
print("Dense retrieval hoàn tất:", len(questions))

del embedder, query_vectors, scores, rows, index
gc.collect()
torch.cuda.empty_cache()



## 9. Rerank trực tiếp



In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

rerank_tokenizer = AutoTokenizer.from_pretrained(RERANK_MODEL_ID)
rerank_model = AutoModelForSequenceClassification.from_pretrained(
    RERANK_MODEL_ID, torch_dtype="auto"
).to("cuda").eval()


def score_pairs(query, passages):
    values = []
    for start in range(0, len(passages), RERANK_BATCH):
        pairs = [[query, passage] for passage in passages[start:start + RERANK_BATCH]]
        inputs = rerank_tokenizer(
            pairs, padding=True, truncation=True, max_length=2304, return_tensors="pt"
        ).to("cuda")
        with torch.inference_mode():
            logits = rerank_model(**inputs, return_dict=True).logits.view(-1)
        values.extend(logits.float().cpu().tolist())
    return values


node_lookup = {node["node_id"]: node for node in nodes}
candidate_records = [json.loads(line) for line in candidates_path.read_text(encoding="utf-8").splitlines() if line]
reranked_path = OUTPUT_DIR / "reranked.jsonl"
temporary = reranked_path.with_suffix(".jsonl.tmp")
with temporary.open("w", encoding="utf-8") as output:
    for number, record in enumerate(candidate_records, 1):
        passages = [embedding_text(node_lookup[hit["node_id"]]) for hit in record["hits"]]
        rerank_scores = score_pairs(record["question"], passages)
        if len(rerank_scores) != len(record["hits"]) or any(not math.isfinite(float(score)) for score in rerank_scores):
            raise ValueError("Reranker trả scores không hợp lệ")
        hits = [
            {**hit, "rerank_score": float(score)}
            for hit, score in zip(record["hits"], rerank_scores)
        ]
        hits.sort(key=lambda hit: (-hit["rerank_score"], -hit["dense_score"], hit["node_id"]))
        output.write(json.dumps({**record, "hits": hits[:RERANK_TOP_K]}, ensure_ascii=False) + "\n")
        if number % 10 == 0:
            print(f"Reranked {number:,}/{len(candidate_records):,}", flush=True)
temporary.replace(reranked_path)
print("Rerank hoàn tất")

del rerank_model, rerank_tokenizer, candidate_records
gc.collect()
torch.cuda.empty_cache()



## 10. Sinh đáp án và resume trực tiếp



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SYSTEM_PROMPT = (
    "Bạn là trợ lý pháp luật Việt Nam. Chỉ trả lời từ căn cứ được cung cấp. "
    "Nêu căn cứ pháp lý, trình bày quy định theo đúng thứ tự và kết luận trực tiếp. "
    "Không tự tạo số Điều, Khoản, mức tiền hoặc thủ tục. Nếu căn cứ không đủ, nói rõ chưa đủ căn cứ."
)


def build_context(record):
    blocks, used, seen = [], 0, set()
    for hit in record["hits"][:CONTEXT_TOP_K]:
        node = node_lookup[hit["node_id"]]
        if node["node_id"] in seen:
            continue
        block = f"[Nguồn {len(blocks) + 1}] {' > '.join(node.get('path', []))}\n{node['content']}"
        size = len(block.split())
        if blocks and used + size > MAX_CONTEXT_WORDS:
            break
        blocks.append(block)
        used += size
        seen.add(node["node_id"])
    return "\n\n".join(blocks)


generator_tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL_ID)
model_kwargs = {"device_map": "auto"}
if USE_4BIT:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16
    )
else:
    model_kwargs["torch_dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
generator_model = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL_ID, **model_kwargs).eval()

reranked_records = [json.loads(line) for line in reranked_path.read_text(encoding="utf-8").splitlines() if line]
state_path = OUTPUT_DIR / "generation_state.json"
submission_path = OUTPUT_DIR / "submission.json"
completed = json.loads(state_path.read_text(encoding="utf-8")) if state_path.exists() else {}


def atomic_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


for number, record in enumerate(reranked_records, 1):
    qid, question = str(record["id"]), record["question"]
    if qid in completed and completed[qid].get("question") == question and completed[qid].get("answer"):
        continue
    context = build_context(record)
    prompt = f"{SYSTEM_PROMPT}\n\nCĂN CỨ:\n{context or '[Không có căn cứ phù hợp]'}\n\nCÂU HỎI:\n{question}\n\nTRẢ LỜI:"
    messages = [{"role": "user", "content": prompt}]
    if generator_tokenizer.chat_template:
        text = generator_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = prompt
    inputs = generator_tokenizer(text, return_tensors="pt").to(generator_model.device)
    with torch.inference_mode():
        output = generator_model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, repetition_penalty=1.05
        )
    answer = generator_tokenizer.decode(
        output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True
    ).strip()
    if not answer:
        raise RuntimeError(f"Model trả answer rỗng cho câu {qid}")
    completed = {**completed, qid: {"question": question, "answer": answer}}
    atomic_json(state_path, completed)
    if number % 5 == 0:
        print(f"Generated {number:,}/{len(reranked_records):,}", flush=True)

expected_ids = [str(record["id"]) for record in reranked_records]
missing = [qid for qid in expected_ids if qid not in completed or not completed[qid].get("answer")]
assert not missing, f"Còn thiếu {len(missing)} answers"
submission = {qid: {"answer": completed[qid]["answer"]} for qid in expected_ids}
atomic_json(submission_path, submission)
print("Generation hoàn tất:", len(submission))



## 11. Kiểm tra format và tải submission.zip



In [ ]:
assert list(submission) == list(questions), "ID hoặc thứ tự submission không khớp đề"
assert all(
    set(record) == {"answer"} and isinstance(record["answer"], str) and record["answer"].strip()
    for record in submission.values()
), "Submission sai format"

zip_path = OUTPUT_DIR / "submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(submission_path, arcname="submission.json")
print("Submission:", zip_path, f"({zip_path.stat().st_size / 1024**2:.1f} MB)")
first_id = next(iter(submission))
print(json.dumps({first_id: submission[first_id]}, ensure_ascii=False, indent=2)[:3000])

from google.colab import files
files.download(str(zip_path))
